# Session 5 · de-code skills and MCP

*Democratizing Coding with AI — Fri 9 Oct*

<nils.holmberg@iko.lu.se>

## Topics

<!-- table of contents: update when a section heading is renamed or moved -->
- 1 Skills
    - 1.1 From prompts to skills
    - 1.2 Anatomy of a skill
    - 1.3 How agents use skills
    - 1.4 Official and custom skills
    - 1.5 A real skill: from HTML app to PWA
    - 1.6 Use the skill
- 2 MCP: Model Context Protocol
    - 2.1 Why MCP
    - 2.2 Anatomy of MCP
    - 2.3 Skills and MCP
    - 2.4 Build a minimal MCP server
    - 2.5 Existing MCP servers
- 3 Putting it together
    - 3.1 A skill that uses MCP
    - 3.2 The week in one table
    - 3.3 Your turn
- [Slides](https://de-code-ai.netlify.app/coding/skill-school/sessions/05-skills-mcp-slides.html)
- [Notebook](https://colab.research.google.com/github/nils-holmberg/de-code/blob/main/jnb/skill-school/05-skills-mcp.ipynb)

In [ ]:
#| echo: false
import matplotlib.pyplot as plt

# illustrative token counts: what sits in the agent's context all the time,
# and what is only loaded when a task needs it
parts = ["AGENTS.md rules", "10 skill descriptions", "MCP tool list",
         "1 skill body", "1 skill script"]
tokens = [1500, 800, 4000, 2500, 1200]
colors = ["#2f9e44", "#2f9e44", "#2f9e44", "#b2f2bb", "#b2f2bb"]

fig, ax = plt.subplots()
ax.barh(parts[::-1], tokens[::-1], color=colors[::-1], height=0.62)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xlabel("tokens (illustrative)")
ax.set_title("Always loaded (dark) vs loaded on demand (light)")
plt.tight_layout()
plt.show()

# 1 Skills

## 1.1 From prompts to skills

This week we have given AI agents four kinds of text. Each one does a different job:

| Text | What it says | When the agent reads it | Session |
|------|--------------|-------------------------|---------|
| Prompt | Do this, now | Once, when you send it | 1–2 |
| Spec | What to build and how to check it | When you point the agent to it | 3–4 |
| AGENTS.md | Rules that always apply | At the start of every task | 4 |
| **Skill** | **How to do a kind of task, step by step** | **Only when the task needs it** | 5 |

A **skill** is a reusable procedure: a folder with instructions, and optionally scripts and
templates, that teaches an agent how to do one kind of task well — the way *you* want it
done. Write it once, and every later task of that kind starts from your best practice
instead of from scratch.

## 1.2 Anatomy of a skill

A skill is a folder. Only one file is required, `SKILL.md`; the rest is optional:

```text
skill-html-pwa/
├── SKILL.md                  required: name, description, instructions
├── references/netlify.md     optional: a longer document it reads only when needed
├── scripts/validate_pwa.py   optional: code the agent can run to check its work
└── agents/openai.yaml        optional: extra settings for one agent (OpenAI Codex)
```

This is the real skill we look at in section 1.5. A skill can also have an `assets/` folder
with templates and other files to copy. Here is a minimal `SKILL.md`:

In [ ]:
#| eval: false
minimal_skill = """---
name: tsv-summary
description: Summarise a tab-separated data file. Use when the user asks what is in
  a .tsv file, or wants a quick overview of its columns before an analysis.
---

# TSV summary

1. Read the file with pandas (sep="\\t").
2. Report the number of rows and columns, and the missing values per column.
3. For each numeric column: mean, SD, min and max, rounded to two decimals.
4. For each text column: the five most frequent values.
5. End with two sentences on what the data seems to be about.
"""

`SKILL.md` has two parts. The **front matter** between the `---` lines holds the skill's
`name` and a `description` of what it does and when to use it. The **body** is plain
Markdown: the steps, rules and checks.

## 1.3 How agents use skills

An agent may have dozens of skills, but it does not read them all. At the start it only
sees each skill's **name and description** — a few lines each, so they cost little of the
context window (see the chart above). When your request matches a description, the agent
reads that skill's body, and opens its scripts or references only if a step needs them.
This is called *progressive disclosure*.

So the **description is the trigger**. Write it as *what it does* plus *"Use when …"*, with
the words people actually use when they ask for that task. There are two ways to use a
skill:

- **Automatically:** ask in your own words ("give me an overview of survey.tsv") and the
  agent picks the matching skill.
- **By name:** type a slash command with the skill name, e.g. `/tsv-summary survey.tsv`.

In Antigravity, a project's skills live in `.agents/skills/<name>/SKILL.md` inside the
project folder; skills for all your projects are configured under `~/.gemini/config`. Type
`/skills` to see which skills the agent has, and `/skills reload` after adding or editing
one.

## 1.4 Official and custom skills

| | Official skills | Custom skills |
|---|---|---|
| Written by | The company behind the agent or tool | You, your team or your course |
| Examples | Antigravity's built-in skills such as `/grill-me`; Anthropic's [public skills](https://github.com/anthropics/skills) for Word, Excel, PowerPoint and PDF files | The HTML-to-PWA skill below |
| Good for | General tasks that are the same for everyone | *Your* data, conventions, and quality checks |
| Updated | By the publisher | By you, when your practice changes |
| Trust | Comes from a known source | You know what is in it |

Both follow the same open format, [Agent Skills](https://agentskills.io/), so a skill
written for one agent often works in others. A skill can include scripts that run on your
computer, so treat skills from others like any other software: read `SKILL.md` and its
scripts before you install it (session 4, *Safety first!*).

## 1.5 A real skill: from HTML app to PWA

A **progressive web app** (PWA) is a web page you can install like an app: it gets an icon
on your phone or computer, opens in its own window, and keeps working offline after the
first visit. The skill below turns an existing single-page HTML app — like the ones we built
in sessions 3 and 4 — into a PWA, and prepares it for a static web host such as Netlify.

An agent used this skill to turn a regex trainer page into the
[de-code regex trainer](https://de-code-ai.netlify.app/app-building/pwa/) — open it on your
phone and try *Add to home screen*. This is its `SKILL.md`, unchanged:

In [ ]:
#| eval: false
skill_md = """---
name: skill-html-pwa
description: Convert an existing single-page HTML web app into a branded, installable, mobile-compatible PWA and prepare or verify it on a static live host such as Netlify. Use when adapting a supplied HTML app rather than building an unrelated site from scratch.
---

# HTML app to hosted PWA

Turn the supplied single-page app into a real progressive web app while preserving its purpose and behavior. Treat branding, mobile usability, installability, offline behavior, host path, and live verification as one delivery problem.

## Establish the deployment shape

Before editing, inspect:

- the source HTML and its scripts, styles, assets, and existing behavior;
- supplied brand names, copy, colors, logos, favicons, and icons;
- the requested output directory;
- the host's publish root and the app's public URL path; and
- existing manifests, service workers, redirects, headers, and host configuration.

Infer straightforward details from the repository and requested URL. Ask only when a missing choice would materially change the product or deployment. Preserve the original source unless the user explicitly wants it edited in place.

Map the filesystem to the public URL explicitly. For example, if the publish root is `web/` and the target URL is `/app-building/pwa/`, the app belongs in `web/app-building/pwa/`.

## Preserve and brand the app

Keep the original app's useful interactions, content, saved state, and accessibility semantics unless the user requests a change. Apply requested branding consistently to:

- the document title, description, theme color, and mobile metadata;
- the visible primary title and subtitle;
- the header or app chrome;
- the favicon, application icons, and installed-app name; and
- focus, selection, match, and other primary highlight states.

Prefer supplied brand assets. Inspect them before use. Generate raster sizes from the supplied vector or highest-quality source when needed; do not merely rename a small bitmap as a larger icon.

## Build a real PWA

Create physical, host-served files rather than Blob URLs:

- `index.html`
- `manifest.webmanifest` (or a correctly linked JSON manifest)
- `sw.js`
- at least actual 192×192 and 512×512 PNG icons
- the supplied SVG or other logo assets used by the page

For an app hosted below the domain root, default to relative paths:

    {
      "id": "./",
      "start_url": "./",
      "scope": "./"
    }

Link resources with `./...` and register `./sw.js` with scope `./`. Avoid root-relative paths such as `/icons/icon.png` unless the asset intentionally lives at the origin root. A service worker cannot be registered from a `blob:` URL.

The manifest should include the requested name, a short name that fits launcher UI, description, `standalone` display mode, theme and background colors, and valid icon entries. Use `purpose: "any maskable"` only when the artwork stays legible inside the maskable safe area.

## Design offline behavior safely

Cache the local application shell during service-worker installation and provide a cached navigation fallback. Use a site-specific cache namespace and increment its version when shell files change.

When removing old caches, delete only caches owned by this app. Cache storage is origin-wide even though service-worker control is scoped; never delete every cache on a host that may contain sibling apps.

Browser storage is origin-wide as well: `localStorage`, `sessionStorage` and IndexedDB are shared by every app on the host. Give new keys and database names an app-specific prefix, such as `regexcraft_pattern`, and never clear all storage. Keep the source app's existing keys unless you also migrate their values, so users do not lose saved state.

Prefer local CSS, JavaScript, fonts, and icons for dependable offline startup. If the source relies on CDNs and retaining them is proportionate, explain that the first visit must be online and use runtime caching deliberately. Do not claim complete offline behavior until it has been tested with network access disabled.

## Make the existing UI mobile-compatible

Use a responsive viewport without disabling zoom. Keep primary body text readable, controls touch-friendly, focus visible, and text selectable where useful. Verify that:

- the title and essential brand copy remain visible;
- navigation condenses without becoming ambiguous;
- controls do not clip or overlap;
- intentional horizontal toolbars scroll without causing page-wide overflow;
- editors and results remain usable on a narrow portrait screen; and
- the layout also remains coherent at desktop width and 200% text enlargement.

Do not replace a functioning app with a landing page or put promotional content ahead of its primary activity.

## Prepare the host

The app requires HTTPS in production. Preserve the user's hosting provider and deployment workflow. Do not publish, push, or mutate a live host unless requested.

For Netlify-specific placement, MIME headers, and verification, read [references/netlify.md](references/netlify.md).

## Validate before handoff

Run the bundled validator:

    python scripts/validate_pwa.py /path/to/pwa

Then serve the app over HTTP; `file://` cannot validate service workers. Check at least one desktop and one narrow mobile viewport, exercise the app's primary interaction, and inspect the actual screenshots. Verify the manifest and every icon URL, service-worker registration and cache creation, reload behavior, and offline startup after an online load.

For a live URL, check response status and MIME types rather than assuming the upload path is correct. Confirm that HTML is served as HTML, the service worker as JavaScript, the manifest as `application/manifest+json` or another browser-accepted JSON type, and icons with their declared image types.

Report what was created, the public-path assumption, what was directly verified, and any remaining host-side action. Distinguish "prepared for deployment" from "deployed and live."
"""

What to notice:

- The **description** says what the skill does *and* when to use it ("Use when adapting a
  supplied HTML app …"), so the agent can pick it by itself.
- The **body** is a procedure with checks, not code: inspect first, keep what works, build
  the PWA files, make it work offline and on mobile, then validate.
- The **Netlify details** live in `references/netlify.md`, and the skill says to read that
  file only when the host is Netlify — progressive disclosure (section 1.3).
- A **script** checks what instructions alone cannot: `scripts/validate_pwa.py` tests the
  manifest, the icon sizes and the service worker.
- It asks for **honest reporting**: the agent must say whether the app is only *prepared*
  for deployment or actually *deployed and live*.

## 1.6 Use the skill

Download [skill-html-pwa.zip](https://de-code-ai.netlify.app/agentic-ai/skill-html-pwa.zip)
(or browse the files on
[GitHub](https://github.com/nils-holmberg/de-code/tree/main/web/agentic-ai/skill-html-pwa)),
read `SKILL.md` and the script first (section 1.4), and unzip it into `.agents/skills/` in a
project folder. Check that Antigravity lists it with `/skills`, then try it both ways — by
name and in your own words — on the word-cloud app from sessions 3 and 4:

In [ ]:
#| eval: false
by_name = """
/skill-html-pwa Turn wordcloud.html into an installable PWA that will be hosted
at https://<your-site>.netlify.app/apps/wordcloud/
"""

own_words = """
Make my word-cloud page installable on my phone, so it also opens without internet.
"""

A good run ends with the validator's output and a clear statement: is the app only
*prepared* for deployment, or *deployed and live*? The word-cloud app loads its libraries
from a CDN, so also check what the agent says about offline use — the skill tells it not to
claim offline support it has not tested.

# 2 MCP: Model Context Protocol

## 2.1 Why MCP

A skill teaches the agent *how* to do something with the tools it already has: reading and
writing files, running commands. Some tasks need more: opening a web page in a real
browser, reading issues on GitHub, querying a database, checking a calendar. The
[Model Context Protocol](https://modelcontextprotocol.io/) (MCP) is an open standard for
giving an agent such new **tools and data**.

Before MCP, every AI app needed its own connector for every service. With MCP, a service
is wrapped once as an **MCP server**, and every agent that speaks MCP — Antigravity, Claude,
ChatGPT and many code editors — can use it. It is often compared to USB-C: one plug for
everything.

## 2.2 Anatomy of MCP

| Part | What it is | Example |
|------|-----------|---------|
| Host | The AI app you use | Antigravity |
| Client | The connection inside the host, one per server | Antigravity's link to the GitHub server |
| Server | A small program that offers capabilities | The GitHub MCP server |

A server can offer three kinds of things:

- **Tools** — actions the agent can call, with named inputs: `search_issues(query)`,
  `take_screenshot(url)`.
- **Resources** — data the agent can read: a file, a database table, a codebook.
- **Prompts** — ready-made prompt templates the user can pick.

Servers run in one of two ways (the *transport*): **locally**, as a program on your
computer that the host starts and talks to (stdio), or **remotely**, at a web address
(HTTP). One tool call goes like this:

1. You ask: *"Which issues are open in my repo?"*
2. The agent sees a matching tool in the server's tool list and calls it, with inputs.
3. Antigravity asks for your approval (unless you allowed that tool before).
4. The server does the work and sends the result back.
5. The agent reads the result and answers you — or calls the next tool.

## 2.3 Skills and MCP

| | Skill | MCP server |
|---|---|---|
| Adds | Know-how: steps, rules, checks | Abilities and data: new tools |
| Is | Text (+ optional scripts) | A running program |
| Context cost | Small until used | Its tool list is loaded while it is on |
| Write your own | Easy: a Markdown file | Some code (a few lines in Python) |
| Use when | You repeat a kind of task | The agent cannot reach something it needs |

They work well together: a skill can say *which* MCP tools to use, and in what order.

## 2.4 Build a minimal MCP server

A server in Python takes a few lines with the official
[MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk). Each tool is an
ordinary function: its name, type hints and docstring tell the agent what it does and which
inputs it takes. This server offers two of the session 1 analyses as tools, plus the
dataset's codebook as a resource:

In [ ]:
#| eval: false
server_code = '''
"""komp014 MCP server: quick analyses of a tab-separated data file."""
import pandas as pd
from scipy import stats
from mcp.server import MCPServer

mcp = MCPServer("komp014")


@mcp.tool()
def describe_tsv(url: str) -> dict:
    """Rows, columns and the mean of every numeric column of a .tsv file."""
    df = pd.read_csv(url, sep="\\t")
    means = df.select_dtypes("number").mean().round(2)
    return {"rows": len(df), "columns": list(df.columns), "means": means.to_dict()}


@mcp.tool()
def group_difference(url: str, group: str, outcome: str) -> dict:
    """Welch t-test and Cohen's d for the outcome between the two levels of group."""
    df = pd.read_csv(url, sep="\\t")
    a, b = [g[outcome] for _, g in df.groupby(group)]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    sd = ((a.var() * (len(a) - 1) + b.var() * (len(b) - 1)) / (len(a) + len(b) - 2)) ** 0.5
    d = (a.mean() - b.mean()) / sd
    return {"t": round(float(t), 2), "p": round(float(p), 4), "d": round(float(d), 2)}


@mcp.resource("komp014://codebook")
def codebook() -> str:
    """What the columns of the komp014 toy dataset mean."""
    return ("IV_CAT_GROUP: category1/category2. Items 1-5: IV1_ITEM1, IV1_ITEM2, "
            "IV2_ITEM1, IV2_ITEM2, DV_ITEM1, DV_ITEM2. Indices are item means: "
            "IV_INDEX1, IV_INDEX2, DV_INDEX (outcome).")


if __name__ == "__main__":
    mcp.run()  # local server: talks to the host over stdin/stdout
'''

with open("komp014_server.py", "w") as f:
    f.write(server_code)

To use it, you need Python with the `mcp` package on your own computer (MCP servers run next
to the agent, not in Colab). Try the server in the **MCP Inspector**, a web page where you
can call its tools by hand, then add it to Antigravity and check that it is listed:

In [ ]:
#| eval: false
!pip install "mcp[cli]" pandas scipy
!mcp dev komp014_server.py
!agy mcp add komp014 python /full/path/to/komp014_server.py
!agy mcp list

Then ask the agent something only the server can answer quickly:

In [ ]:
#| eval: false
mcp_prompt = """
Use the komp014 tools: describe
https://raw.githubusercontent.com/nils-holmberg/de-code/main/csv/komp014-simulated-data-n100.tsv
and test whether DV_INDEX differs between the IV_CAT_GROUP levels.
Explain the result in two sentences.
"""

The answer should report a Cohen's *d* of size 0.58 (its sign shows which group scores
higher) — the same effect as in session 1, now computed by a tool the agent called.

## 2.5 Existing MCP servers

Many MCP servers are ready to use. Most are added with a single `agy mcp add` command
(or in the Antigravity app's MCP settings) — local ones as a command, remote ones as a URL:

| Server | What the agent can do | Add it |
|--------|----------------------|--------|
| [Filesystem](https://github.com/modelcontextprotocol/servers) | Read and write files in folders you choose | `agy mcp add fs npx -y @modelcontextprotocol/server-filesystem ~/projects` |
| [Fetch](https://github.com/modelcontextprotocol/servers) | Download a web page as text | `agy mcp add fetch uvx mcp-server-fetch` |
| [Git](https://github.com/modelcontextprotocol/servers) | Read history, diffs and branches of a repository | `agy mcp add git uvx mcp-server-git` |
| [Memory](https://github.com/modelcontextprotocol/servers) | Remember facts between conversations | `agy mcp add memory npx -y @modelcontextprotocol/server-memory` |
| [Time](https://github.com/modelcontextprotocol/servers) | Current time and time zone conversion | `agy mcp add time uvx mcp-server-time` |
| [Playwright](https://github.com/microsoft/playwright-mcp) | Open and click through web pages in a browser | `agy mcp add playwright npx -y @playwright/mcp@latest` |
| [Chrome DevTools](https://github.com/ChromeDevTools/chrome-devtools-mcp) | Inspect a page in Chrome: console errors, layout, speed | `agy mcp add devtools npx -y chrome-devtools-mcp@latest` |
| [GitHub](https://github.com/github/github-mcp-server) | Issues, pull requests and code on GitHub (needs a token) | `agy mcp add --header "Authorization: Bearer <token>" github https://api.githubcopilot.com/mcp/` |
| [Context7](https://github.com/upstash/context7) | Up-to-date documentation for code libraries (needs a key) | `agy mcp add --header "Authorization: Bearer <key>" context7 https://mcp.context7.com/mcp` |

`npx` needs [Node.js](https://nodejs.org/) and `uvx` needs [uv](https://docs.astral.sh/uv/)
on your computer. The same safety rules as in session 4 apply, with one more: an MCP server
is a program that runs with *your* permissions.

- Prefer servers from the service itself or the MCP project, and read what each tool does.
- Give tokens and folders the smallest access that works (read-only if you can).
- Keep tool approval on, and turn servers off (`agy mcp disable <name>`) when you do not
  need them — their tool lists also take up context.

# 3 Putting it together

## 3.1 A skill that uses MCP

Skills and MCP combine naturally. The PWA skill already asks the agent to "render the live
URL at desktop and mobile sizes" — but an agent can only do that if it has a browser. A
browser MCP server from section 2.5 gives it one. Add a short section to the skill, and the
agent uses those tools when they are there:

In [ ]:
#| eval: false
skill_md += """
## With browser tools
If a Chrome DevTools or Playwright MCP server is available, open the live URL at
390 px and 1280 px width, check the console for errors, and confirm that the
service worker is registered.
"""

The skill holds the *procedure*, the server provides the *tool*, and AGENTS.md still sets the
*rules* — each part stays small and can be reused on its own. Antigravity can also bundle
skills, rules and MCP servers into one installable **plugin** (`/plugin`), which is how
teams share a complete setup.

## 3.2 The week in one table

| Session | You made | What the AI got from you |
|---------|----------|--------------------------|
| 1 Data analysis | An analysis, step by step | Prompts |
| 2 Data generation | Simulated data and personas | A detailed prompt, then an analysis spec |
| 3 App building | App specs: Shiny, word clouds | Specs with acceptance checks |
| 4 Agentic coding | Apps built by an agent, spec by spec; a first agent in Python | AGENTS.md rules and specs that it updates |
| 5 Skills and MCP | A reusable skill, a small MCP server | Procedures and new tools |

The common thread is *specifications as code*: the clearer and more reusable the text you
give an AI, the more you can hand over — and the easier it is to check the result.

## 3.3 Your turn

1. Pick a task you repeat in your own work — a report, a data check, a type of analysis.
2. Write it as a skill: a `SKILL.md` with a clear *"Use when …"* description and five to
   ten steps, saved in `.agents/skills/<name>/` in a project folder.
3. Use it twice in Antigravity, once by name and once in your own words, and improve the
   description until the agent picks it up by itself.
4. Pick one MCP server from the table, add it, and use it in a small task — then turn it off
   again if you do not need it.